# 42 – Jobs per Acre in TOCs / TODs (LODES)

This notebook calculates **jobs per acre** for each TOC and TOD using LODES 2022 jobs.

**Goals:**
- Load LODES jobs + geometry layer.
- Ensure CRS matches project CRS and compute area.
- Spatially join jobs to TOCs/TODs.
- Aggregate jobs and area by TOC/TOD.
- Compute jobs per acre at the TOC/TOD level.


In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

JOBS_DIR = DATA_DIR / "jobs" / "lodes_processed"
BOUNDARIES_DIR = DATA_DIR / "boundaries" / "processed"

lodes_jobs_path = JOBS_DIR / "lodes_jobs_2022_with_geometry.gpkg"
toc_clean_path = BOUNDARIES_DIR / "toc_clean.gpkg"

lodes_jobs_path, toc_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/jobs/lodes_processed/lodes_jobs_2022_with_geometry.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/boundaries/processed/toc_clean.gpkg'))

In [2]:
lodes_jobs = gpd.read_file(lodes_jobs_path)
toc = gpd.read_file(toc_clean_path)

print("LODES CRS:", lodes_jobs.crs)
print("TOC CRS:", toc.crs)

LODES CRS: EPSG:2868
TOC CRS: EPSG:2868


## CRS consistency

For **jobs per acre**, we need a **projected CRS** (meters or feet) so area calculations are meaningful.

We will:
- Choose `PROJECT_CRS` (e.g., same as parcels).
- Reproject LODES, TOC, and TOD layers if necessary.


In [3]:
# they are the same, but just because I am an anxious person...

PROJECT_CRS = "EPSG:2868"

if toc.crs != PROJECT_CRS:
    toc = toc.to_crs(PROJECT_CRS)

lodes_jobs.crs, toc.crs

(<Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - Ellipsoid: GRS 1980
 - Prime Meridian: Greenwich,
 <Projected CRS: EPSG:2868>
 Name: NAD83(HARN) / Arizona Central (ft)
 Axis Info [cartesian]:
 - X[east]: Easting (foot)
 - Y[north]: Northing (foot)
 Area of Use:
 - name: United States (USA) - Arizona - counties Coconino; Maricopa; Pima; Pinal; Santa Cruz; Yavapai.
 - bounds: (-113.35, 31.33, -110.44, 37.01)
 Coordinate Operation:
 - name: SPCS83 Arizona Central zone (international foot)
 - method: Transverse Mercator
 Datum: NAD83 (High Accuracy Reference Network)
 - 

In [4]:
if "jobs_total" not in lodes_jobs.columns and "C000" in lodes_jobs.columns:
    lodes_jobs["jobs_total"] = lodes_jobs["C000"].fillna(0)

if "area_sq_ft" not in lodes_jobs.columns:
    lodes_jobs["area_sq_ft"] = lodes_jobs.geometry.area
    lodes_jobs["area_acres"] = lodes_jobs["area_sq_ft"] / 43560.0

In [5]:
# give TOCs a stable ID index:

toc = toc.reset_index().rename(columns={"index": "TOC_ID"})
# but keep APPLICABIL as the human-readable label

In [6]:
toc.head(10)

,TOC_ID,OBJECTID,APPLICABIL,geometry
0,0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89..."
1,1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89..."
2,2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90..."
3,3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915..."
4,4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91..."
5,5,6,TOD District - 19North,"POLYGON ((646874.486 919833.609, 646872.259 91..."
6,6,7,50th Street Station Area,"POLYGON ((684257.247 888198.254, 684382.127 88..."
7,7,8,Capitol Extension Area,"POLYGON ((641505.991 895598.434, 643697.045 89..."
8,8,9,I-10 West Extension Area,"POLYGON ((602060.385 899622.267, 602090.536 90..."
9,9,10,Northwest Extension Area II,"POLYGON ((633784.013 939340.634, 644284.495 93..."


In [7]:
lodes_jobs.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,C000,CE01,CE02,CE03,area_sq_ft,area_acres,jobs_total,jobs_per_acre,geometry
0,040130101021,04,013,010102,1,455,66,114,275,2.584523e+10,593324.922484,455,0.000767,"POLYGON ((827801.708 1091626.91, 827865.552 10..."
1,040130101021,04,013,010102,2,455,66,114,275,1.276018e+08,2929.334492,455,0.155325,"POLYGON ((725542.865 1023666.739, 723744.814 1..."
2,040130101021,04,013,010102,3,455,66,114,275,3.515639e+09,80707.955066,455,0.005638,"POLYGON ((712735.735 1097380.426, 712738.472 1..."
3,040130101021,04,013,010103,1,455,66,114,275,1.653396e+08,3795.674584,455,0.119873,"POLYGON ((744671.443 1002586.263, 744681.625 1..."
4,040130101021,04,013,010103,2,455,66,114,275,2.399420e+08,5508.309491,455,0.082602,"POLYGON ((765755.318 1002785.693, 765592.367 1..."


In [8]:
bg_x_toc = gpd.overlay(
    lodes_jobs,
    toc[["TOC_ID", "APPLICABIL", "geometry"]],
    how="intersection",
    keep_geom_type=False
)

bg_x_toc.head()

,GEOID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,C000,CE01,CE02,CE03,area_sq_ft,area_acres,jobs_total,jobs_per_acre,TOC_ID,APPLICABIL,geometry
0,040130405141,04,013,082010,2,42,19,6,17,8.798369e+06,201.982763,42,0.207939,8,I-10 West Extension Area,"POLYGON ((602105.618 904625.198, 602104.11 904..."
1,040130405151,04,013,082018,2,609,81,174,354,7.210200e+06,165.523427,609,3.679238,8,I-10 West Extension Area,"POLYGON ((602081.728 900941.945, 602081.692 90..."
2,040130405151,04,013,082028,1,609,81,174,354,5.902619e+06,135.505489,609,4.494283,8,I-10 West Extension Area,"POLYGON ((602062.613 899197.31, 602059.378 898..."
3,040130405362,04,013,104205,1,497,121,204,172,7.973872e+06,183.054923,497,2.715032,9,Northwest Extension Area II,"POLYGON ((633781.057 938842.466, 633779.119 93..."
4,040130405362,04,013,104205,2,497,121,204,172,3.385554e+06,77.721621,497,6.394617,9,Northwest Extension Area II,"POLYGON ((635092.755 938836.829, 636414.206 93..."


In [9]:
# Area of each BG–TOC slice in square feet / acres
bg_x_toc["intersect_sqft"] = bg_x_toc.geometry.area
bg_x_toc["intersect_acres"] = bg_x_toc["intersect_sqft"] / 43560.0

# Fraction of each BG’s area that lies inside this TOC
bg_x_toc["area_weight"] = (
    bg_x_toc["intersect_sqft"] /
    bg_x_toc["area_sq_ft"]
)

# sanity check:

bg_x_toc["area_weight"].describe()

count    2.540000e+02
mean     5.844380e-01
std      4.605951e-01
min      5.920639e-07
25%      4.947704e-03
50%      9.450335e-01
75%      9.999754e-01
max      1.000000e+00
Name: area_weight, dtype: float64

In [10]:
# area weight jobs into TOCs

bg_x_toc["jobs_contrib"] = (
    bg_x_toc["jobs_total"].fillna(0) *
    bg_x_toc["area_weight"]
)

In [11]:
toc_jobs = (
    bg_x_toc
    .groupby("TOC_ID", as_index=False)
    .agg({
        "jobs_contrib": "sum",
        "intersect_acres": "sum"
    })
)

toc_jobs = toc_jobs.rename(columns={
    "jobs_contrib": "jobs_total",
    "intersect_acres": "acres_contributing"
})

# so total_jobs = total jobs allocated to that TOC and/or station area
# acres_contributing is the total area in acres of block groups overlapping with the TOC

In [12]:
tocs_with_jobs = toc.merge(
    toc_jobs,
    on="TOC_ID",
    how="left"
)

# true TOC area based on its own geometry
tocs_with_jobs["toc_area_sqft"] = tocs_with_jobs.geometry.area
tocs_with_jobs["toc_acres"] = tocs_with_jobs["toc_area_sqft"] / 43560.0

# jobs per acre
tocs_with_jobs["jobs_per_acre"] = (
    tocs_with_jobs["jobs_total"] / tocs_with_jobs["toc_acres"]
)

tocs_with_jobs.head(11)

,TOC_ID,OBJECTID,APPLICABIL,geometry,jobs_total,acres_contributing,toc_area_sqft,toc_acres,jobs_per_acre
0,0,1,TOD District - Gateway,"POLYGON ((663431.678 894322.427, 663434.469 89...",4984.867123,2418.944362,1.053692e+08,2418.944362,2.060761
1,1,2,TOD District - Eastlake Garfield,"POLYGON ((654751.939 894390.665, 654764.736 89...",6847.046904,1220.462979,5.316337e+07,1220.462979,5.610205
2,2,3,TOD District - Midtown,"POLYGON ((654767.886 907514.625, 654767.631 90...",7099.922548,1283.000217,5.588749e+07,1283.000217,5.533844
3,3,4,TOD District - Uptown,"POLYGON ((649448.684 915530.44, 654791.762 915...",441.878230,1332.064285,5.802472e+07,1332.064285,0.331724
4,4,5,TOD District - Solano,"POLYGON ((646830.036 919545.608, 646829.436 91...",2006.609086,1100.265433,4.792756e+07,1100.265433,1.823750
5,5,6,TOD District - 19North,"POLYGON ((646874.486 919833.609, 646872.259 91...",2635.120908,1780.947386,7.757807e+07,1780.947386,1.479617
6,6,7,50th Street Station Area,"POLYGON ((684257.247 888198.254, 684382.127 88...",1628.549532,617.105659,2.688112e+07,617.105659,2.639012
7,7,8,Capitol Extension Area,"POLYGON ((641505.991 895598.434, 643697.045 89...",3529.734836,1116.466203,4.863327e+07,1116.466203,3.161524
8,8,9,I-10 West Extension Area,"POLYGON ((602060.385 899622.267, 602090.536 90...",6465.078133,7735.922063,3.369768e+08,7735.922063,0.835722
9,9,10,Northwest Extension Area II,"POLYGON ((633784.013 939340.634, 644284.495 93...",1633.684459,1740.704483,7.582509e+07,1740.704483,0.938519


In [13]:
tocs_with_jobs["jobs_per_acre"].describe()

count    11.000000
mean      2.539864
std       1.787908
min       0.331724
25%       1.209068
50%       2.060761
75%       3.342676
max       5.610205
Name: jobs_per_acre, dtype: float64

In [14]:
total_jobs_bg = lodes_jobs["jobs_total"].sum()
total_jobs_toc = tocs_with_jobs["jobs_total"].sum()

total_jobs_toc / total_jobs_bg

0.02778230022110838

In [15]:
toc_jobs_path = JOBS_DIR / "toc_jobs_2022.gpkg"
tocs_with_jobs.to_file(toc_jobs_path, driver="GPKG")

In [16]:
toc_jobs_path = JOBS_DIR / "toc_jobs_2022.csv"
tocs_with_jobs.to_csv(toc_jobs_path, index=False)